In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Modular Addition — matched learning-rate sweep

Reviewer-targeted sweep using the same one-layer `AdditionTransformer` as the rebuttal notebook.

- Matched conditions: **LayerNorm + raw head** vs **sphere + bounded head**
- Learning rates: `1e-4, 3e-4, 6e-4, 1e-3`
- Seeds: `1, 2, 3` (the same seed gives the same train/test split in both conditions and at every LR)
- Fixed: `p=113`, 30% train split, `d_model=128`, 4 heads, MLP 512, AdamW, WD=1, 100k maximum epochs
- Logs every 50 epochs; Drive recovery checkpoints every 500 epochs

Run all cells. Completed runs are skipped; interrupted runs resume from their recovery checkpoint.


In [ ]:
import os, json, math, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Reviewer-targeted matched sweep. Edit only this block if needed.
P = 113
FRAC_TRAIN = 0.30
D_MODEL = 128
NUM_HEADS = 4
MLP_DIM = 512
WEIGHT_DECAY = 1.0
TAU = 10.0

LRS = [1e-4, 3e-4, 6e-4, 1e-3]
SEEDS = [1, 2, 3]
MAX_EPOCHS = 100_000
LOG_EVERY = 5
RECOVERY_EVERY = 500
POST_99_GRACE = 1_000
STABLE_99_LOGS = 3

SAVE_ROOT = Path('/content/drive/MyDrive/grokking_rebuttal_uniform_attention_lr_sweep_v1')
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

CONDITIONS = [
    # Intervention B only. Baseline is the existing LayerNorm/raw learned-attention runs.
    dict(name='layernorm_raw_uniform', model_type='layernorm', bounded_head=False, tau=None,
         uniform_attention=True),
]

print(f'{len(LRS)} LRs x {len(SEEDS)} seeds x {len(CONDITIONS)} conditions = '
      f'{len(LRS) * len(SEEDS) * len(CONDITIONS)} runs')


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_dataset(p, frac_train, seed):
    # Same seed => identical split for every architecture and LR.
    pairs = [(a, b) for a in range(p) for b in range(p)]
    random.Random(seed).shuffle(pairs)
    n_train = int(len(pairs) * frac_train)

    def build(part):
        x = torch.tensor([[a, b, p] for a, b in part], dtype=torch.long)
        y = torch.tensor([(a + b) % p for a, b in part], dtype=torch.long)
        return x, y

    train_x, train_y = build(pairs[:n_train])
    test_x, test_y = build(pairs[n_train:])
    return train_x, train_y, test_x, test_y


class AdditionTransformer(nn.Module):
    # Kept identical to the component-ablation implementation.
    def __init__(self, p, model_type, bounded_head, tau, uniform_attention=False):
        super().__init__()
        self.model_type = model_type
        self.bounded_head = bounded_head
        self.tau = tau
        self.uniform_attention = uniform_attention
        self.d_head = D_MODEL // NUM_HEADS

        self.tok_embed = nn.Embedding(p + 1, D_MODEL)
        self.pos_embed = nn.Embedding(3, D_MODEL)
        self.W_Q = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_K = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_V = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_O = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.mlp_in = nn.Linear(D_MODEL, MLP_DIM)
        self.mlp_out = nn.Linear(MLP_DIM, D_MODEL)
        self.unembed = nn.Linear(D_MODEL, p, bias=False)

        if model_type == 'layernorm':
            self.input_norm = nn.LayerNorm(D_MODEL)
            self.norm1 = nn.LayerNorm(D_MODEL)
            self.norm2 = nn.LayerNorm(D_MODEL)

    def norm(self, h, which):
        if self.model_type == 'spherical':
            return F.normalize(h, dim=-1)
        if which == 0:
            return self.input_norm(h)
        return self.norm1(h) if which == 1 else self.norm2(h)

    def forward(self, x):
        _, seq_len = x.shape
        pos = torch.arange(seq_len, device=x.device).unsqueeze(0)
        h0 = self.norm(self.tok_embed(x) + self.pos_embed(pos), 0)

        q = self.W_Q(h0).view(-1, seq_len, NUM_HEADS, self.d_head).transpose(1, 2)
        k = self.W_K(h0).view(-1, seq_len, NUM_HEADS, self.d_head).transpose(1, 2)
        v = self.W_V(h0).view(-1, seq_len, NUM_HEADS, self.d_head).transpose(1, 2)
        if self.uniform_attention:
            # Intervention B: override pre-softmax scores with zeros -> uniform [1/3,1/3,1/3].
            scores = torch.zeros(
                x.shape[0], NUM_HEADS, seq_len, seq_len, device=x.device, dtype=q.dtype
            )
        else:
            scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn = F.softmax(scores, dim=-1)
        attn_out = self.W_O(
            (attn @ v).transpose(1, 2).contiguous().view(-1, seq_len, D_MODEL)
        )

        h1 = self.norm(h0 + attn_out, 1)
        h2 = self.norm(h1 + self.mlp_out(F.relu(self.mlp_in(h1))), 2)
        h_final = h2[:, 2, :]

        if self.bounded_head:
            # Paper-aligned normalized unembedding; h is already unit norm for sphere.
            return F.linear(h_final, F.normalize(self.unembed.weight, dim=1)) * float(self.tau)
        return self.unembed(h_final)


def atomic_torch_save(obj, path):
    tmp = Path(str(path) + '.tmp')
    torch.save(obj, tmp)
    os.replace(tmp, path)


def atomic_json_dump(obj, path):
    tmp = Path(str(path) + '.tmp')
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


def first_epoch(history, key, threshold):
    for epoch, value in zip(history['epoch'], history[key]):
        if value >= threshold:
            return int(epoch)
    return None


def test_rise_onset(history):
    # First logged test accuracy >=10% that gains >=5 percentage points within 100 epochs.
    epochs = np.asarray(history['epoch'])
    acc = np.asarray(history['test_acc'])
    for i in range(len(epochs)):
        if acc[i] < 0.10:
            continue
        future = (epochs > epochs[i]) & (epochs <= epochs[i] + 100)
        if future.any() and acc[future].max() >= acc[i] + 0.05:
            return int(epochs[i])
    return None


In [ ]:
@torch.no_grad()
def evaluate(model, x, y, criterion):
    model.eval()
    logits = model(x)
    return criterion(logits, y).item(), (logits.argmax(-1) == y).float().mean().item()


def train_one(cfg, seed, paths):
    train_x, train_y, test_x, test_y = make_dataset(P, FRAC_TRAIN, seed)
    train_x, train_y = train_x.to(DEVICE), train_y.to(DEVICE)
    test_x, test_y = test_x.to(DEVICE), test_y.to(DEVICE)

    set_seed(seed)
    model = AdditionTransformer(
        P, cfg['model_type'], cfg['bounded_head'], cfg['tau'],
        uniform_attention=cfg.get('uniform_attention', False),
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg['lr'], weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999)
    )
    criterion = nn.CrossEntropyLoss()
    history = {k: [] for k in ['epoch', 'train_loss', 'test_loss', 'train_acc', 'test_acc']}
    start_epoch = 0
    test99_epoch = None
    stable99 = 0
    stop_reason = 'max_epochs'

    if paths['recovery'].exists():
        state = torch.load(paths['recovery'], map_location=DEVICE)
        model.load_state_dict(state['model_state_dict'])
        optimizer.load_state_dict(state['optimizer_state_dict'])
        history = state['history']
        start_epoch = state['epoch'] + 1
        test99_epoch = state.get('test99_epoch')
        stable99 = state.get('stable99', 0)
        print('resume:', cfg['run_name'], 'seed', seed, 'from epoch', start_epoch)

    bar = tqdm(range(start_epoch, MAX_EPOCHS), desc=f"{cfg['run_name']} seed{seed}")
    started = time.time()
    for epoch in bar:
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(train_x), train_y)
        if not torch.isfinite(loss):
            stop_reason = f'non_finite_loss_epoch_{epoch}'
            break
        loss.backward()
        optimizer.step()

        should_log = epoch % LOG_EVERY == 0 or epoch == MAX_EPOCHS - 1
        if should_log:
            train_loss, train_acc = evaluate(model, train_x, train_y, criterion)
            test_loss, test_acc = evaluate(model, test_x, test_y, criterion)
            for key, value in [
                ('epoch', epoch), ('train_loss', train_loss), ('test_loss', test_loss),
                ('train_acc', train_acc), ('test_acc', test_acc),
            ]:
                history[key].append(value)

            bar.set_postfix(train=f'{train_acc:.3f}', test=f'{test_acc:.3f}')
            if test99_epoch is None and test_acc >= 0.99:
                test99_epoch = epoch
            stable99 = stable99 + 1 if test_acc >= 0.99 else 0

            if stable99 >= STABLE_99_LOGS and epoch >= test99_epoch + POST_99_GRACE:
                stop_reason = 'stable_test_99_post_grace'
                break

        if epoch % RECOVERY_EVERY == 0:
            atomic_torch_save(
                dict(
                    epoch=epoch,
                    model_state_dict=model.state_dict(),
                    optimizer_state_dict=optimizer.state_dict(),
                    history=history,
                    test99_epoch=test99_epoch,
                    stable99=stable99,
                    config=cfg,
                    seed=seed,
                ),
                paths['recovery'],
            )

    elapsed = time.time() - started
    summary = dict(
        condition=cfg['name'], model_type=cfg['model_type'],
        head_type='bounded' if cfg['bounded_head'] else 'raw',
        attention='uniform' if cfg.get('uniform_attention', False) else 'learned',
        lr=cfg['lr'], seed=seed, weight_decay=WEIGHT_DECAY,
        stop_reason=stop_reason, last_epoch=history['epoch'][-1],
        train_epoch_95=first_epoch(history, 'train_acc', 0.95),
        train_epoch_99=first_epoch(history, 'train_acc', 0.99),
        test_rise_onset=test_rise_onset(history),
        test_epoch_95=first_epoch(history, 'test_acc', 0.95),
        test_epoch_99=first_epoch(history, 'test_acc', 0.99),
        peak_test_acc=max(history['test_acc']),
        final_train_acc=history['train_acc'][-1],
        final_test_acc=history['test_acc'][-1],
        elapsed_seconds=elapsed,
    )
    summary['gap_99'] = (
        None if summary['train_epoch_99'] is None or summary['test_epoch_99'] is None
        else summary['test_epoch_99'] - summary['train_epoch_99']
    )
    # Table 1 reported the first >95% checkpoint, so report the 95% gap alongside it.
    summary['gap_95'] = (
        None if summary['train_epoch_95'] is None or summary['test_epoch_95'] is None
        else summary['test_epoch_95'] - summary['train_epoch_95']
    )
    # Negative => test rise begins before the training threshold is reached.
    summary['onset_minus_train_99'] = (
        None if summary['test_rise_onset'] is None or summary['train_epoch_99'] is None
        else summary['test_rise_onset'] - summary['train_epoch_99']
    )

    atomic_torch_save(
        dict(model_state_dict=model.state_dict(), summary=summary, config=cfg), paths['model']
    )
    atomic_json_dump(dict(summary=summary, history=history, config=cfg), paths['json'])
    if paths['recovery'].exists():
        paths['recovery'].unlink()
    return summary


In [ ]:
# Seed-first ordering keeps matched train/test splits adjacent.
rows = []
for seed in SEEDS:
    for lr in LRS:
        for base in CONDITIONS:
            cfg = dict(base, lr=lr)
            lr_tag = f'{lr:.0e}'.replace('+', '')
            cfg['run_name'] = f"{cfg['name']}__lr{lr_tag}"
            run_dir = SAVE_ROOT / cfg['run_name']
            run_dir.mkdir(parents=True, exist_ok=True)
            stem = f"{cfg['run_name']}__seed{seed}"
            paths = {
                'json': run_dir / f'{stem}.json',
                'model': run_dir / f'{stem}.pt',
                'recovery': run_dir / f'{stem}_recovery.pt',
            }

            if paths['json'].exists() and paths['model'].exists():
                with open(paths['json']) as f:
                    summary = json.load(f)['summary']
                print('skip completed:', stem)
            else:
                summary = train_one(cfg, seed, paths)
                print('saved:', stem, '| Test99:', summary['test_epoch_99'],
                      '| peak:', f"{summary['peak_test_acc']:.4f}")

            rows.append(summary)
            pd.DataFrame(rows).to_csv(SAVE_ROOT / 'summary_incremental.csv', index=False)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

df = pd.DataFrame(rows).sort_values(['lr', 'condition', 'seed'])
df.to_csv(SAVE_ROOT / 'per_run_results.csv', index=False)

summary = (
    df.groupby(['condition', 'model_type', 'head_type', 'attention', 'lr'], dropna=False)
      .agg(
          seeds=('seed', 'count'),
          reached_99=('test_epoch_99', lambda x: int(x.notna().sum())),
          mean_rise_onset=('test_rise_onset', 'mean'),
          std_rise_onset=('test_rise_onset', 'std'),
          mean_train_95=('train_epoch_95', 'mean'),
          mean_train_99=('train_epoch_99', 'mean'),
          mean_test_95=('test_epoch_95', 'mean'),
          mean_test_99=('test_epoch_99', 'mean'),
          std_test_99=('test_epoch_99', 'std'),
          mean_gap_95=('gap_95', 'mean'),
          mean_gap_99=('gap_99', 'mean'),
          mean_onset_minus_train_99=('onset_minus_train_99', 'mean'),
          mean_peak_test_acc=('peak_test_acc', 'mean'),
      )
      .reset_index()
)
summary.to_csv(SAVE_ROOT / 'lr_sweep_summary.csv', index=False)

print('\nPER-RUN RESULTS')
print(df[['condition', 'attention', 'lr', 'seed',
          'train_epoch_95', 'test_epoch_95', 'gap_95',
          'train_epoch_99', 'test_rise_onset', 'test_epoch_99', 'gap_99',
          'onset_minus_train_99', 'peak_test_acc', 'stop_reason']].to_string(index=False))
print('\nLR SWEEP SUMMARY (means use successful seeds for threshold metrics)')
print(summary.to_string(index=False))

# For each architecture: maximize successes first, then minimize mean Test99.
ranking = summary.copy()
ranking['_test99_for_rank'] = ranking['mean_test_99'].fillna(np.inf)
best = (ranking.sort_values(['condition', 'reached_99', '_test99_for_rank'],
                            ascending=[True, False, True])
               .groupby('condition', as_index=False).head(1)
               .drop(columns='_test99_for_rank'))
best.to_csv(SAVE_ROOT / 'best_lr_by_condition.csv', index=False)
print('\nBEST LR PER CONDITION')
print(best.to_string(index=False))
print('\nSaved to:', SAVE_ROOT)


In [ ]:
"""Build the matched learned- vs uniform-attention comparison from the two sweep runs.

Reads the per-run JSON files written by both notebooks (each contains the full
history), recomputes every threshold with one definition, and prints the table.

Run in Colab after mounting Drive.
"""

import json
from pathlib import Path

import numpy as np
import pandas as pd

DRIVE = Path('/content/drive/MyDrive')
LEARNED_ROOT = DRIVE / 'grokking_rebuttal_addition_lr_sweep_v1'
UNIFORM_ROOT = DRIVE / 'grokking_rebuttal_uniform_attention_lr_sweep_v1'
OUT_DIR = DRIVE / 'grokking_rebuttal_intervention_b_comparison'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Only these conditions from the learned sweep are the Intervention B baseline.
BASELINE_CONDITIONS = {'layernorm_raw'}


def first_epoch(history, key, threshold):
    for epoch, value in zip(history['epoch'], history[key]):
        if value >= threshold:
            return int(epoch)
    return None


def test_rise_onset(history):
    """First logged test acc >=10% that gains >=5 points within 100 epochs."""
    epochs = np.asarray(history['epoch'])
    acc = np.asarray(history['test_acc'])
    for i in range(len(epochs)):
        if acc[i] < 0.10:
            continue
        future = (epochs > epochs[i]) & (epochs <= epochs[i] + 100)
        if future.any() and acc[future].max() >= acc[i] + 0.05:
            return int(epochs[i])
    return None


def log_every(history):
    e = np.asarray(history['epoch'])
    return int(np.median(np.diff(e))) if len(e) > 1 else None


def load(root, keep_conditions=None):
    rows = []
    if not root.exists():
        print(f'!! missing directory: {root}')
        return rows
    for path in sorted(root.rglob('*.json')):
        if path.name.endswith('.tmp'):
            continue
        with open(path) as f:
            blob = json.load(f)
        if 'history' not in blob or 'summary' not in blob:
            continue
        cfg, hist, summ = blob.get('config', {}), blob['history'], blob['summary']
        cond = summ.get('condition', cfg.get('name'))
        if keep_conditions is not None and cond not in keep_conditions:
            continue

        attention = summ.get('attention')
        if attention is None:  # older runs predate the field
            attention = 'uniform' if cfg.get('uniform_attention') else 'learned'

        r = dict(
            condition=cond,
            attention=attention,
            lr=summ.get('lr', cfg.get('lr')),
            seed=summ.get('seed'),
            log_every=log_every(hist),
            train95=first_epoch(hist, 'train_acc', 0.95),
            train99=first_epoch(hist, 'train_acc', 0.99),
            test95=first_epoch(hist, 'test_acc', 0.95),
            test99=first_epoch(hist, 'test_acc', 0.99),
            rise_onset=test_rise_onset(hist),
            peak_test=max(hist['test_acc']),
            last_epoch=hist['epoch'][-1],
            stop_reason=summ.get('stop_reason'),
        )
        for a, b, name in [('test95', 'train95', 'gap95'),
                           ('test99', 'train99', 'gap99'),
                           ('rise_onset', 'train99', 'onset_minus_train99')]:
            r[name] = None if r[a] is None or r[b] is None else r[a] - r[b]
        rows.append(r)
    return rows


rows = load(LEARNED_ROOT, BASELINE_CONDITIONS) + load(UNIFORM_ROOT)
if not rows:
    raise SystemExit('No runs found. Check the two SAVE_ROOT paths.')

df = pd.DataFrame(rows).sort_values(['lr', 'attention', 'seed'])
df.to_csv(OUT_DIR / 'per_run_merged.csv', index=False)

print('\nLOGGING RESOLUTION PER GROUP (epochs between evaluations)')
print(df.groupby(['attention'])['log_every'].agg(['min', 'max']).to_string())

print('\nPER-RUN')
cols = ['attention', 'lr', 'seed', 'train95', 'test95', 'gap95',
        'train99', 'rise_onset', 'test99', 'gap99', 'onset_minus_train99',
        'peak_test', 'stop_reason']
print(df[cols].to_string(index=False))

agg = (df.groupby(['attention', 'lr'], dropna=False)
         .agg(seeds=('seed', 'count'),
              reached99=('test99', lambda x: int(x.notna().sum())),
              train95=('train95', 'mean'),
              test95=('test95', 'mean'),
              gap95=('gap95', 'mean'),
              train99=('train99', 'mean'),
              rise_onset=('rise_onset', 'mean'),
              test99=('test99', 'mean'),
              test99_sd=('test99', 'std'),
              gap99=('gap99', 'mean'),
              onset_minus_train99=('onset_minus_train99', 'mean'),
              peak_test=('peak_test', 'mean'))
         .reset_index())
agg.to_csv(OUT_DIR / 'summary_by_attention_lr.csv', index=False)
print('\nSUMMARY BY ATTENTION x LR')
print(agg.to_string(index=False, float_format=lambda v: f'{v:.4g}'))

# Side-by-side speedup table, the one that answers the reviewer directly.
wide = agg.pivot(index='lr', columns='attention',
                 values=['rise_onset', 'test99', 'gap99', 'onset_minus_train99'])
out = pd.DataFrame(index=wide.index)
for metric in ['rise_onset', 'test99', 'gap99', 'onset_minus_train99']:
    for att in ['learned', 'uniform']:
        if (metric, att) in wide.columns:
            out[f'{metric}_{att}'] = wide[(metric, att)]
    a, b = f'{metric}_learned', f'{metric}_uniform'
    if a in out and b in out and metric in ('rise_onset', 'test99'):
        out[f'{metric}_speedup'] = out[a] / out[b]
out = out.reset_index()
out.to_csv(OUT_DIR / 'learned_vs_uniform.csv', index=False)
print('\nLEARNED vs UNIFORM (matched LR, matched betas, same onset definition)')
print(out.to_string(index=False, float_format=lambda v: f'{v:.4g}'))
print('\nSaved to:', OUT_DIR)

In [ ]:
"""Threshold-crossing profiles for the learned- vs uniform-attention runs.

Answers: when does each curve cross 10/25/50/75/90/95/99%, how do train and test
interleave, and what does the test curve do between its onset and 99%.

Reads the per-run JSON files (which contain the full history) from both sweeps.
Run in Colab after mounting Drive.
"""

import json
from pathlib import Path

import numpy as np
import pandas as pd

DRIVE = Path('/content/drive/MyDrive')
LEARNED_ROOT = DRIVE / 'grokking_rebuttal_addition_lr_sweep_v1'
UNIFORM_ROOT = DRIVE / 'grokking_rebuttal_uniform_attention_lr_sweep_v1'
OUT_DIR = DRIVE / 'grokking_rebuttal_intervention_b_comparison'
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_CONDITIONS = {'layernorm_raw'}
THRESHOLDS = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]


def first_epoch(history, key, threshold):
    for epoch, value in zip(history['epoch'], history[key]):
        if value >= threshold:
            return int(epoch)
    return None


def test_rise_onset(history):
    epochs = np.asarray(history['epoch'])
    acc = np.asarray(history['test_acc'])
    for i in range(len(epochs)):
        if acc[i] < 0.10:
            continue
        future = (epochs > epochs[i]) & (epochs <= epochs[i] + 100)
        if future.any() and acc[future].max() >= acc[i] + 0.05:
            return int(epochs[i])
    return None


def test_acc_at(history, epoch):
    """Test accuracy at the last logged point <= epoch."""
    if epoch is None:
        return None
    e = np.asarray(history['epoch'])
    a = np.asarray(history['test_acc'])
    m = e <= epoch
    return float(a[m][-1]) if m.any() else None


def load(root, keep=None):
    rows = []
    if not root.exists():
        print(f'!! missing directory: {root}')
        return rows
    for path in sorted(root.rglob('*.json')):
        if path.name.endswith('.tmp'):
            continue
        blob = json.load(open(path))
        if 'history' not in blob or 'summary' not in blob:
            continue
        cfg, hist, summ = blob.get('config', {}), blob['history'], blob['summary']
        cond = summ.get('condition', cfg.get('name'))
        if keep is not None and cond not in keep:
            continue
        att = summ.get('attention')
        if att is None:
            att = 'uniform' if cfg.get('uniform_attention') else 'learned'

        r = dict(attention=att, lr=summ.get('lr', cfg.get('lr')), seed=summ.get('seed'))
        for t in THRESHOLDS:
            tag = f'{int(t * 100)}'
            r[f'tr{tag}'] = first_epoch(hist, 'train_acc', t)
            r[f'te{tag}'] = first_epoch(hist, 'test_acc', t)
        r['onset'] = test_rise_onset(hist)
        r['tr99'] = r['tr99']
        # How far the test curve has already climbed when training finishes.
        r['test_acc_at_tr99'] = test_acc_at(hist, r['tr99'])
        # Shape of the test climb itself.
        r['climb_10_to_95'] = (None if r['te10'] is None or r['te95'] is None
                               else r['te95'] - r['te10'])
        r['climb_50_to_95'] = (None if r['te50'] is None or r['te95'] is None
                               else r['te95'] - r['te50'])
        r['climb_95_to_99'] = (None if r['te95'] is None or r['te99'] is None
                               else r['te99'] - r['te95'])
        # Does the test curve start moving before training is done?
        r['onset_minus_tr99'] = (None if r['onset'] is None or r['tr99'] is None
                                 else r['onset'] - r['tr99'])
        r['te10_minus_tr99'] = (None if r['te10'] is None or r['tr99'] is None
                                else r['te10'] - r['tr99'])
        r['te50_minus_tr99'] = (None if r['te50'] is None or r['tr99'] is None
                                else r['te50'] - r['tr99'])
        rows.append(r)
    return rows


rows = load(LEARNED_ROOT, BASELINE_CONDITIONS) + load(UNIFORM_ROOT)
if not rows:
    raise SystemExit('No runs found. Check the two SAVE_ROOT paths.')

df = pd.DataFrame(rows).sort_values(['lr', 'attention', 'seed'])
df.to_csv(OUT_DIR / 'curve_profiles_per_run.csv', index=False)

fmt = lambda v: f'{v:.4g}'

print('\nTRAIN CURVE: first epoch at each threshold')
tr_cols = [f'tr{int(t * 100)}' for t in THRESHOLDS]
print(df.groupby(['attention', 'lr'])[tr_cols].mean().reset_index()
        .to_string(index=False, float_format=fmt))

print('\nTEST CURVE: first epoch at each threshold')
te_cols = [f'te{int(t * 100)}' for t in THRESHOLDS]
print(df.groupby(['attention', 'lr'])[te_cols].mean().reset_index()
        .to_string(index=False, float_format=fmt))

print('\nINTERLEAVING: negative => test reaches this level before training hits 99%')
mix = ['onset', 'tr99', 'onset_minus_tr99', 'te10_minus_tr99', 'te50_minus_tr99',
       'test_acc_at_tr99']
print(df.groupby(['attention', 'lr'])[mix].mean().reset_index()
        .to_string(index=False, float_format=fmt))

print('\nSHAPE OF THE TEST CLIMB (epochs spent between levels)')
shape = ['climb_10_to_95', 'climb_50_to_95', 'climb_95_to_99']
print(df.groupby(['attention', 'lr'])[shape].mean().reset_index()
        .to_string(index=False, float_format=fmt))

print('\nPER-RUN (test thresholds + interleaving)')
print(df[['attention', 'lr', 'seed'] + te_cols +
         ['onset', 'tr99', 'onset_minus_tr99', 'test_acc_at_tr99']]
      .to_string(index=False, float_format=fmt))

print('\nSaved to:', OUT_DIR / 'curve_profiles_per_run.csv')